In [1]:
from argparse import ArgumentParser, Namespace
#import Orange
import Spectral_Preprocessing as sp
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import json

In [2]:
parser = ArgumentParser()
parser.add_argument("--range-ir-1", dest= "wn_range_ir_1", default = (948,1860))
parser.add_argument("--range-ir-2", dest= "wn_range_ir_2", default = (2698,3002))

parser.add_argument("--range-min-ir", dest= "range_min_ir", type = int, default = 400)
parser.add_argument("--range-max-ir", dest= "range_max_ir", type = int, default = 4000)
parser.add_argument("--interpolate-interval-ir", dest = "interpolate_interval_ir", type = int, default = 1) 

parser.add_argument("--finger-print-exl", dest= "finger_print_exl", type = bool, default = False)
parser.add_argument("--trans-chip", dest= "trans_chip", type = bool, default = False)

parser.add_argument("--background-rule-ir", dest= "background_rule", type = str, default = 'mean')
parser.add_argument("--threshold-mean-intensity-ir", dest= "ths_mean_int_ir", type = float, default = 0)
parser.add_argument("--threshold-total-variation-intensity-ir", dest= "ths_tot_var_int_ir", type = float, default = 0)

parser.add_argument("--range-min-raman", dest= "range_min_raman", type = float, default = 400)
parser.add_argument("--range-max-raman", dest= "range_max_raman", type = float, default = 4000)
parser.add_argument("--interpolate-interval-raman", dest = "interpolate_interval_raman", type = float, default = 1) 

parser.add_argument("--rubberband", dest="rubberband", action="store_true", default=False,)
parser.add_argument("--minmax", dest="minmax", action="store_true", default=False,)
parser.add_argument("--no-interpolate", dest = "no_interpolate", action="store_true", default = False)
parser.add_argument("--nitrogen-calibration", dest="nitrogen_calibration", action="store_true", default=True,)
parser.add_argument("--si-peak", dest= "si_peak", type = float, default = 520)

parser.add_argument("--smooth", dest = "smooth", type = int, default = 0)
parser.add_argument("--baseline", dest = "baseline", type = int, default = 5)

###Sample Code #1: Specify IR and Raman interpolation intervals along with min & max values
args: Namespace = parser.parse_args([]) ###Do not alter this section.
###PLEASE TYPE IN YOUR ARGUMENTS HERE
args.wn_range_ir_1 = (980, 1800) #(675.0,1800)
args.wn_range_ir_2 = (2720, 2970) #(1800,4000)
args.range_min_ir = 675.0 #674.9918 ###enter an integer
args.range_max_ir = 4000.0 #3999.809 ###enter an integer [max value is not included]
args.interpolate_interval_ir = 2 #1.929 ###enter an integer
args.finger_print_exl = False
args.trans_chip = True
args.background_rule = 'total_var'
args.ths_mean_int_ir = 0.043
args.ths_tot_var_int_ir = 0.4543#0.59

args.range_min_raman = 600 #355 #.555542 ###enter an integer or float #530
args.range_max_raman = 3500 #3895.935059 ##enter an integer or float [max value is not included] 
args.interpolate_interval_raman = 2.0 #168021680216802 ###enter an integer or float

args.rubberband = 'specy'#'specy' ###True for rubberband, False for iModPoly
args.minmax = 'snv' ###False for SVT normalization, True for minmax
args.no_interpolate = False ###use sample code 2 if using True for this option.
args.nitrogen_calibration = True ##True for calibration, False for no calibration.
args.si_peak = 520 ###enter an integer or float

# Rebecca's data

In [3]:
#directory_name = '/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
directory_name = 'MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
ir_export_filename =  "no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes
file_list_tmp = [] ###option to list out the files you want included. if left as-is, all files in directory will be included.

path = 'Reference_library_reb/'
dirs = os.listdir(path)
dirs = [x for x in dirs if x not in ['.DS_Store', 'Archive for reference data', 'Processed']]

file_list = []
for d in dirs:
    #print(os.listdir(path + d + '/Data/Processed/mIRage OPTIR+Raman/'))
    path_2 = path + d + '/Data/Processed/mIRage OPTIR+Raman/'
    dirs_2 = os.listdir(path_2)
    #file_list = file_list + [path_2 + dd for dd in dirs_2 if dd[-4:] == "xlsx"]
    file_list = file_list + [path_2 + dd for dd in dirs_2 if (dd[-9:] == "data.xlsx" or dd[-12:] == "spectra.xlsx")]
file_list = file_list + [d + '/' + dd for dd in dirs_2 if (dd[-9:] == "data.xlsx" or dd[-12:] == "spectra.xlsx")]

polymer_list = np.array(['sucrose', 'PE', 'PSL', 'NaNO3', 'NH2SO4', 'PP'])

wn_list = []
df_list = []
for i, f in enumerate(file_list):
    df = pd.read_excel(f, sheet_name=None)#['Raman', 'Raman spectra']
    print(f)
    if 'Raman' in df.keys():
        df_raman = df['Raman']
    elif 'Raman spectra' in df.keys():
        df_raman = df['Raman spectra']
    df_raman.columns = ['wavenumber'] + list(range(df_raman.shape[1]-1))#.set_index('X value [cmâ»Â¹]')
    df_raman = df_raman.set_index('wavenumber')
    df_raman = df_raman.T.reset_index(drop=True)
    df_raman['polymer'] = polymer_list[[p in f for p in polymer_list]][0]#polymer[i]
    df_raman['map_x'] = 0
    df_raman['map_y'] = 0
    
    if 'O-PTIR' in df.keys():
        df_ir = df['O-PTIR']
    elif 'PTIR spectra' in df.keys():
        df_ir = df['PTIR spectra']
    df_ir.columns = ['wavenumber'] + list(range(df_ir.shape[1]-1))#.set_index('X value [cmâ»Â¹]')
    df_ir = df_ir.set_index('wavenumber')
    df_ir = df_ir.T.reset_index(drop=True)
    df_ir['polymer'] = polymer_list[[p in f for p in polymer_list]][0]#polymer[i]
    df_ir['map_x'] = 0
    df_ir['map_y'] = 0
    
    df_list.append([df_ir,df_raman])
    
pol_table = pd.DataFrame({'file':[polymer_list[[p in f for p in polymer_list]][0] for f in file_list]})

args.si_peak = [520]*len(df_list)
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, _, rng, ir_bg_label, saturated_wn_list = sp.df_to_interpolated_processed_csv(args, directory_name=None, ir_export_filename=None, raman_export_filename=None, file_list = df_list, transpose = True)


Reference_library_reb/20230708 0.05 M sucrose/Data/Processed/mIRage OPTIR+Raman/20240509_50mMsucrose_RP_mIRage_allccdata.xlsx
Reference_library_reb/20230423 1.0 and 3.0 um PSL/Data/Processed/mIRage OPTIR+Raman/20230424_1and3umPSL_allccdata.xlsx
Reference_library_reb/20230708 0.05 M NaNO3/Data/Processed/mIRage OPTIR+Raman/20240509_50mMNaNO3_RP_mIRage_allccdata.xlsx
Reference_library_reb/20230707 200 to 9900 nm PE EtOH iced atomized/Data/Processed/mIRage OPTIR+Raman/20230707_200to9900nmPE_EtOHiced_atomized_mIRage_allccdata.xlsx
Reference_library_reb/20230708 0.05 M NH2SO4/Data/Processed/mIRage OPTIR+Raman/20240509_50mMNH42SO4_RP_mIRage_allccdata.xlsx
Reference_library_reb/20230906 1 to 5 um PP 50-50 EtOH-Omni 2 min sonication iced/Data/Processed/mIRage OPTIR+Raman/20230906_1to5umPP_2minsonic_iced_30minMPSstg2_allspectra.xlsx


In [5]:
df_ir

wavenumber,948,950,952,954,956,958,960,962,964,966,...,2990,2992,2994,2996,2998,3000,3002,polymer,map_x,map_y
0,-6.388178,-17.854105,-18.496817,-5.954295,-2.273004,-0.943817,-0.981189,1.907225,-5.569495,-4.061456,...,-9.851234,0.111614,-14.935507,-19.064146,-10.028702,-4.058281,3.384930,PP,0,0
1,-51.345272,-23.935825,-21.097054,-14.709726,-13.940348,-20.117092,-15.200953,-10.693293,-4.115892,-3.247877,...,-23.011889,-20.141701,-22.507959,-29.007256,-16.372087,-24.426981,-26.885336,PP,0,0
2,-17.644272,-14.052928,-10.014669,-4.656581,-9.858616,1.530119,-0.241036,-7.248718,-5.061189,-3.473799,...,0.276670,-6.262599,-11.339095,-7.894164,-14.042855,-13.072949,-8.783506,PP,0,0
3,6.703174,4.307558,2.332650,7.050187,2.038176,0.873794,0.068287,-2.488892,-2.599187,-1.855537,...,-0.179016,-3.631422,-1.775507,-5.451245,-5.248739,-3.507794,4.209589,PP,0,0
4,0.407098,3.926879,0.494779,-4.039350,-2.981425,0.038774,1.246385,-2.986641,4.345696,9.084301,...,1.634527,9.658705,-0.283538,0.247435,16.466379,6.913344,4.413523,PP,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,-20.645309,-1.190990,-3.219234,2.799243,2.776046,0.508885,-3.512642,-2.143086,-0.067331,0.087619,...,-14.517015,-15.287485,-7.277778,-9.521664,-10.080922,-33.855263,-10.624229,PP,0,0
91,-8.675163,-1.314029,-3.784646,-2.776995,0.498126,0.405069,-1.768945,-3.171202,-2.983779,-1.636365,...,-9.434506,-12.955741,-8.082253,-7.353454,-16.029520,-16.350954,-4.713586,PP,0,0
92,-11.170501,-2.941386,-6.033707,-3.166989,-2.429241,1.202041,4.494313,-1.350115,1.038535,1.010299,...,1.370025,-5.248919,-8.527332,-14.951231,-17.629339,-17.090456,-18.953278,PP,0,0
93,-10.322934,-11.924020,-4.683872,-1.806466,-6.688991,-6.139885,-5.891613,-5.964916,-3.118912,-2.747451,...,-0.939182,-8.039091,-8.089143,-2.967924,0.998594,-1.688344,-16.124058,PP,0,0


In [4]:
id_list = np.array(label_list, dtype='<U20')
for poly in np.unique(label_list):
    n = sum(np.array(label_list) == poly)
    id_list[id_list==poly] = [poly+'_'+str(i+1) for i in range(n)]

In [5]:
material_list = ['PSL', 'PE', 'PP']

material_sheets = ['PSL', 'PE', 'PP']
df_st_in_lib_ir_list = []
for mat, sheet in zip(material_list, material_sheets):
    df_st_in_lib_ir = pd.read_excel('gloves_contamination/MEC_04_017_Raman_and_PTIR_PSL_PP_PE_Library_Sorted.xlsx', sheet_name=sheet)
    df_st_in_lib_ir = df_st_in_lib_ir.iloc[:-1,:2]#['Number']
    df_st_in_lib_ir['id'] = [mat+ '_' + str(int(x)) for x in df_st_in_lib_ir['ID']]
    df_st_in_lib_ir = df_st_in_lib_ir.loc[:,['id', 'PTIR']]
    df_st_in_lib_ir.columns = ['id', 'in_library']
    df_st_in_lib_ir_list.append(df_st_in_lib_ir)    
df_st_in_lib_ir = pd.concat(df_st_in_lib_ir_list, axis=0)
df_st_in_lib_ir = df_st_in_lib_ir.query('in_library == 1').reset_index(drop=True)

df_st_in_lib_raman_list = []
for mat, sheet in zip(material_list, material_sheets):
    df_st_in_lib_raman = pd.read_excel('gloves_contamination/MEC_04_017_Raman_and_PTIR_PSL_PP_PE_Library_Sorted.xlsx', sheet_name=sheet)
    df_st_in_lib_raman = df_st_in_lib_raman.iloc[:-1,[0,3]]#['Number']
    df_st_in_lib_raman['id'] = [mat+ '_' + str(int(x)) for x in df_st_in_lib_raman['ID']]
    df_st_in_lib_raman = df_st_in_lib_raman.loc[:,['id', 'Raman']]
    df_st_in_lib_raman.columns = ['id', 'in_library']
    df_st_in_lib_raman_list.append(df_st_in_lib_raman)
df_st_in_lib_raman = pd.concat(df_st_in_lib_raman_list, axis=0)
df_st_in_lib_raman = df_st_in_lib_raman.query('in_library == 1').reset_index(drop=True)

In [6]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['polymer'] = label_list
preposs_data_raman['id'] = id_list
#preposs_data_raman = preposs_data_raman.merge(df_st_in_lib_raman).drop('in_library',axis=1).reset_index(drop=True)
preposs_data_raman.to_csv('enviromental_data/processed_data/rebecca_data_raman_final_specy.csv')

preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['polymer'] = label_list
preposs_data_ir['id'] = id_list
#preposs_data_ir = preposs_data_ir.merge(df_st_in_lib_ir).drop('in_library',axis=1).reset_index(drop=True)
preposs_data_ir.to_csv('enviromental_data/processed_data/rebecca_data_ir_final.csv')

In [7]:
preposs_data_raman.shape

(525, 1454)

# slopp

In [7]:
flopp_df = pd.read_csv('gloves_contamination/flopp_flopp_e_for_preprocess.csv')
id_s = flopp_df.loc[:,'file_name']
polymer = flopp_df.loc[:,'polymer']
flopp_df = flopp_df.drop(['Unnamed: 0', 'file_name', 'data_source'], axis=1)
flopp_df = flopp_df.iloc[:,list(range(1,flopp_df.shape[1]))+[0]]
flopp_df['map_x'] = 0
flopp_df['map_y'] = 0
args.si_peak = [520]*len(df_list)
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, _, rng, ir_bg_label, saturated_wn_list = sp.df_to_interpolated_processed_csv(args, directory_name=None, ir_export_filename=None, raman_export_filename=None, file_list = [[flopp_df,flopp_df]], transpose = True)


/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]


In [8]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['polymer'] = label_list
preposs_data_ir['id'] = np.array(id_s)
preposs_data_ir.to_csv('enviromental_data/processed_data/flopp_data_ir_final.csv')

In [9]:
slopp_df = pd.read_csv('processed_data/slopp_slopp_e_pivot.csv')
id_s = slopp_df.loc[:,'file_name']
slopp_df = slopp_df.drop(['Unnamed: 0', 'file_name', 'data_source'], axis=1)
#slopp_df.columns = list(slopp_df.columns[:2]) + list(np.float64(slopp_df.columns[2:-1])) + [slopp_df.columns[1]]
slopp_df = slopp_df.iloc[:,list(range(1,slopp_df.shape[1]))+[0]]
slopp_df['map_x'] = 0
slopp_df['map_y'] = 0
args.si_peak = [520]
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, _, rng, ir_bg_label, saturated_wn_list = sp.df_to_interpolated_processed_csv(args, directory_name=None, ir_export_filename=None, raman_export_filename=None, file_list = [[slopp_df,slopp_df]], transpose = True)


/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]


In [10]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['polymer'] = label_list
preposs_data_raman['id'] = id_s
preposs_data_raman.to_csv('enviromental_data/processed_data/slopp_data_raman_final_specy.csv')

# active enviromental data

## MEC_04_014

In [51]:
files_df_list = []

In [44]:
calibrationLog = pd.read_excel('enviromental_data/mIRageRaman_CalibrationLog_RepeatsRemoved.xlsx')
#calibrationLog = calibrationLog.loc[["Cycled" not in x if type(x) is str else False for x in calibrationLog.iloc[:,-1]], :]
calibrationLog = calibrationLog.rename({'Notes (standard settings: high objective, 10 s integration, 3 averages, 0.5% power, no delay time)':'comments',
                                        'File name':'File_name'}, axis=1)
#duplicated_comments = ['20240913_AMA_RamanCalibration_Si2.ptir',
#                       '20240401_TZ_RamanCalibration_Si.ptir']

#calibrationLog = calibrationLog.query('File_name not in @duplicated_comments').reset_index(drop=True)


In [52]:
directory_name = 'enviromental_data/MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]
files_list


['Stage 4/MEC_04_014_DC_Active_20240808_S4_GridSi_20241009.ptir',
 'Stage 4/MEC_04_014_DC_Active_20240808_S4_GridSi_20240911.ptir',
 'Stage 3/MEC_04_014_DC_Active_20240808_S3_GridSi_20241002.ptir',
 'Stage 3/MEC_04_014_DC_Active_20240808_S3_GridSi_20241011.ptir',
 'Stage 3/MEC_04_014_DC_Active_20240808_S3_GridSi_20241010.ptir',
 'Stage 3/MEC_04_014_DC_Active_20240808_S3_GridSi_20241001.ptir',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210.ptir',
 'Stage 2/20250206/MEC_04_014_DC_Active_20240808_S2_GridSi_20250206.ptir',
 '0.5% Mistakes/MEC_04_014_DC_Active_20240808_S3_GridSi_20241008.ptir',
 '0.5% Mistakes/MEC_04_014_DC_Active_20240808_S4_GridSi_20241008.ptir',
 'Stage 6/MEC_04_014_DC_Active_20240808_S6_ThinSi_20241008.ptir',
 'Stage 6/MEC_04_014_DC_Active_20240808_S6_ThinSi_20241009.ptir',
 'Stage 1/MEC_04_014_DC_Active_20240808_S1_GridSi_20250310.ptir',
 'Stage 1/MEC_04_014_DC_Active_20240808_S1_GridSi_20250306.ptir']

In [53]:
files_df_list.append(pd.DataFrame({'file':[x.split('/')[-1] for x in files_list],
              'type':'active'}))
files_df_list

[                                                 file    type
 0   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 1   MEC_04_014_DC_Active_20240808_S4_GridSi_202409...  active
 2   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 3   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 4   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 5   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 6   MEC_04_014_DC_Active_20240808_S2_GridSi_202412...  active
 7   MEC_04_014_DC_Active_20240808_S2_GridSi_202502...  active
 8   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 9   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 10  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 11  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 12  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active
 13  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active]

In [5]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak = [520]*len(files_list)
args.si_peak

[520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]

In [6]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

In [7]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_1',
       'MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_2',
       'MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_3', ...,
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_60',
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_61',
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_62'],
      dtype='<U55')

In [9]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())

with open('enviromental_data/processed_data/active_data_MEC_04_014_raman_saturation.json', 'w') as f:
    json.dump(saturated_wn_dict, f)

In [17]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir['background'] = ir_bg_label
preposs_data_ir.to_csv('enviromental_data/processed_data/active_data_MEC_04_014_ir.csv')

In [10]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman['wn_saturated'] = [x.shape[0] for x in saturated_wn_list]

preposs_data_raman_norm = preposs_data_raman.copy()
preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]
preposs_data_raman_norm.iloc[np.where(preposs_data_raman.iloc[:,:-4].std(1) == 0)[0], :-4] = 0

ids_no_signal = np.array(preposs_data_raman.query('id not in @ids_sat').iloc[np.where(np.sum(np.abs(np.diff(preposs_data_raman_norm.query('id not in @ids_sat').iloc[:,:-4], axis=1)), 1) < 7)[0],-2])
preposs_data_raman['no_signal'] = [(x in ids_no_signal)*1 for x in ids_file_name]
#preposs_data_raman = preposs_data_raman.loc[np.array(non_nas_idx), :].reset_index(drop=True)
preposs_data_raman['id_file'] = ['/'.join(x.split('/')[:-1]) for x in ids]
#preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_04_014_raman_specy.csv')
preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_04_014_raman_raw.csv')

/var/folders/9g/bj6gc6z17_1fyb7lx57xf04m0000gs/T/ipykernel_88543/1220026886.py:8: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]


## MEC_04_023

In [54]:
directory_name = 'enviromental_data/MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]
files_list


['S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221.ptir',
 'S2/20250220/MEC_04_023_DC_Active_20241212_S2_GridSi_20250220.ptir',
 'S2/20250207/MEC_04_023_DC_Active_20241212_S2_GridSi_20250207.ptir',
 'S3/20250130/MEC_04_023_DC_Active_20241212_S3_GridSi_20250130.ptir',
 'S3/20250122/MEC_04_023_DC_Active_20241212_S3_GridSi_20250122.ptir',
 'S4/20241218/MEC_04_023_DC_Active_20241212_S4_GridSi_20241218.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250227.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250221.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250228.ptir',
 'S6/20250130/MEC_04_023_DC_Active_20241212_S6_GThinSi_20250130.ptir',
 'S6/20250131/MEC_04_023_DC_Active_20241212_S6_ThinSi_20250131.ptir']

In [55]:
files_df_list.append(pd.DataFrame({'file':[x.split('/')[-1] for x in files_list],
              'type':'active'}))
files_df_list

[                                                 file    type
 0   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 1   MEC_04_014_DC_Active_20240808_S4_GridSi_202409...  active
 2   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 3   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 4   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 5   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 6   MEC_04_014_DC_Active_20240808_S2_GridSi_202412...  active
 7   MEC_04_014_DC_Active_20240808_S2_GridSi_202502...  active
 8   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 9   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 10  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 11  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 12  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active
 13  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active,
                                                  file

In [12]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak = [520]*len(files_list)
args.si_peak

[520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]

In [13]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]


In [14]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_1',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_2',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_3',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_4',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_5',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_6',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_7',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_8',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_9',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_10',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_11',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_12',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_13',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_14',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_15',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_16',
       'MEC_04_02

In [15]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())

with open('enviromental_data/processed_data/active_data_MEC_04_023_raman_saturation.json', 'w') as f:
    json.dump(saturated_wn_dict, f)

In [24]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir['background'] = ir_bg_label
preposs_data_ir.to_csv('enviromental_data/processed_data/active_data_MEC_04_023_ir.csv')

In [16]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman['wn_saturated'] = [x.shape[0] for x in saturated_wn_list]

preposs_data_raman_norm = preposs_data_raman.copy()
preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]
preposs_data_raman_norm.iloc[np.where(preposs_data_raman.iloc[:,:-4].std(1) == 0)[0], :-4] = 0

ids_no_signal = np.array(preposs_data_raman.query('id not in @ids_sat').iloc[np.where(np.sum(np.abs(np.diff(preposs_data_raman_norm.query('id not in @ids_sat').iloc[:,:-4], axis=1)), 1) < 7)[0],-2])
preposs_data_raman['no_signal'] = [(x in ids_no_signal)*1 for x in ids_file_name]
#preposs_data_raman = preposs_data_raman.loc[np.array(non_nas_idx), :].reset_index(drop=True)
preposs_data_raman['id_file'] = ['/'.join(x.split('/')[:-1]) for x in ids]
#preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_04_023_raman_specy.csv')
preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_04_023_raman_raw.csv')

/var/folders/9g/bj6gc6z17_1fyb7lx57xf04m0000gs/T/ipykernel_88543/769226507.py:8: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]


## MEC_02_119

In [56]:
directory_name = 'enviromental_data/MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]
files_list


['S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203.ptir',
 'S2/20241204/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241204.ptir',
 'S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522.ptir',
 'S3/20240502/MEC_02_119_DC_Sp_S3_GridSi_FOV1and2_20240502.ptir',
 'S3/20241127/MEC_02_119_DC_Sp_S3_GridSi_20241127.ptir',
 'S3/20240509/MEC_02_119_DC_Sp_S3_GridSi_FOV3_20240509.ptir',
 'S3/20241011/MEC_02_119_DC_Sp_S3_GridSi_20241011.ptir',
 'S3/20240522/MEC_02_119_DC_Sp_S3_GridSi_FOV4,5,6,7,8_20240522.ptir',
 'S4/MEC_02_119_DC_Active_20240418_S4_GridSi_20241030.ptir',
 'S4/MEC_02_119_DC_Sp_S4_GridSi_FOV1_20240424.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240305.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240228.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240306.ptir',
 'S6/ThinSi/MEC_02_119_DC_Active_20240418_S6_ThinSi_20241023.ptir',
 'S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_20240502.ptir']

In [57]:
files_df_list.append(pd.DataFrame({'file':[x.split('/')[-1] for x in files_list],
              'type':'active'}))
files_df_list

[                                                 file    type
 0   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 1   MEC_04_014_DC_Active_20240808_S4_GridSi_202409...  active
 2   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 3   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 4   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 5   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 6   MEC_04_014_DC_Active_20240808_S2_GridSi_202412...  active
 7   MEC_04_014_DC_Active_20240808_S2_GridSi_202502...  active
 8   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 9   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 10  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 11  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 12  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active
 13  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active,
                                                  file

In [18]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak = [520]*len(files_list)
args.si_peak

[520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520, 520]

In [19]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]


In [20]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_1',
       'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_2',
       'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_3', ...,
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_134'], dtype='<U55')

In [21]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())

with open('enviromental_data/processed_data/active_data_MEC_02_119_raman_saturation.json', 'w') as f:
    json.dump(saturated_wn_dict, f)

In [31]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir['background'] = ir_bg_label
preposs_data_ir.to_csv('enviromental_data/processed_data/active_data_MEC_02_119_ir.csv')

In [22]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman['wn_saturated'] = [x.shape[0] for x in saturated_wn_list]

preposs_data_raman_norm = preposs_data_raman.copy()
preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]
preposs_data_raman_norm.iloc[np.where(preposs_data_raman.iloc[:,:-4].std(1) == 0)[0], :-4] = 0

ids_no_signal = np.array(preposs_data_raman.query('id not in @ids_sat').iloc[np.where(np.sum(np.abs(np.diff(preposs_data_raman_norm.query('id not in @ids_sat').iloc[:,:-4], axis=1)), 1) < 7)[0],-2])
preposs_data_raman['no_signal'] = [(x in ids_no_signal)*1 for x in ids_file_name]
#preposs_data_raman = preposs_data_raman.loc[np.array(non_nas_idx), :].reset_index(drop=True)
preposs_data_raman['id_file'] = ['/'.join(x.split('/')[:-1]) for x in ids]
#preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_02_119_raman_specy.csv')
preposs_data_raman.to_csv('enviromental_data/processed_data/active_data_MEC_02_119_raman_raw.csv')

/var/folders/9g/bj6gc6z17_1fyb7lx57xf04m0000gs/T/ipykernel_88543/2413149087.py:8: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]


In [34]:
#_ = plt.plot(preposs_data_ir.query('background=="no_background"').iloc[:,:-4].T)

In [35]:
#_ = plt.plot(preposs_data_raman.query('no_signal==0 and wn_saturated==0').iloc[:,:-5].T)

# passive enviromental data

In [60]:
directory_name = 'enviromental_data/MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'  #'MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'
folders = os.listdir(path)
files_list = []

for folder in folders:
    if folder[-4:] in ["xlsx", "tore"]:
        continue
    dirs = os.listdir(path+folder)
    for d in dirs:
        if d[-4:] in ["tore"]:
            continue
        files = os.listdir(path+folder+'/'+d)
        files_list = files_list+[folder+'/'+d+'/'+f for f in files if f[-5:] == '.ptir']

files_list

['08 August 2024/FHS_20240911/FHS_passive_August2024_B4_20240910.ptir',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_20240911.ptir',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_20240917.ptir',
 '08 August 2024/ICB_20240918/ICB_passive_August2024_E8_20240918.ptir',
 '08 August 2024/DC_20240910/DC_passive_August2024_J10_20240910.ptir',
 '07 July 2024/FHS_20240807/FHS_passive_July2024_D6_20240813.ptir',
 '07 July 2024/FHS_20240807/FHS_passive_July2024_D6_20240807.ptir',
 '07 July 2024/IAO_20240813/IAO_passive_July2024_F10_20240813.ptir',
 '07 July 2024/DC_20240806/DC_passive_July2024_I9_20240806.ptir',
 '07 July 2024/ICB_20240814/ICB_passive_July2024_C4_20240814.ptir',
 '11 November 2024/ICB_20241126/ICB_passive_November2024_C10_20241126.ptir',
 '11 November 2024/DC_20241113/DC_passive_November2024_C5_20241119.ptir',
 '11 November 2024/DC_20241113/DC_passive_November2024_C5_20241113.ptir',
 '11 November 2024/FHS_20241119/FHS_passive_November2024_I5_20241119.ptir',


In [62]:
files_df_list.append(
pd.DataFrame({'file':[x.split('/')[-1] for x in files_list],
                  'type':'passive'})
)
files_df_list

[                                                 file    type
 0   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 1   MEC_04_014_DC_Active_20240808_S4_GridSi_202409...  active
 2   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 3   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 4   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 5   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 6   MEC_04_014_DC_Active_20240808_S2_GridSi_202412...  active
 7   MEC_04_014_DC_Active_20240808_S2_GridSi_202502...  active
 8   MEC_04_014_DC_Active_20240808_S3_GridSi_202410...  active
 9   MEC_04_014_DC_Active_20240808_S4_GridSi_202410...  active
 10  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 11  MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...  active
 12  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active
 13  MEC_04_014_DC_Active_20240808_S1_GridSi_202503...  active,
                                                  file

In [68]:
pd.concat(files_df_list, axis=0).reset_index(drop=True).to_csv('enviromental_data/processed_data/all_file_names.csv')


In [16]:
def ptir_to_data_frame(args, directory_name, ir_export_filename, raman_export_filename, file_list = [], transpose = True):
    directory = os.fsencode(directory_name)
    directory2 = os.fsdecode(directory)
    label_list = []
    merge_frames_exist = False ###creating merge frames on the 1st pass once know XX_ir and XX_raman.
    coord_list = []
    file_names_list= []
    nitro_cal_list = []
    saturated_wn_list = []
    
    ###If no file list is provided, include all files in directory. 
    if file_list == []:
        for file in os.listdir(directory):
            filename = os.fsdecode(file)
            file_list.append(filename)
    
    for i, filename in enumerate(file_list):
        filename2 = '%s%s' %(directory2, filename)
        ir_data, ir_coords, ir_labels = sp.ptir_studio_to_csv(filename2, "IR")
        raman_data, raman_coords, raman_labels = sp.ptir_studio_to_csv(filename2, "Raman")
        check = sp.check_matching_coordinates(ir_coords, raman_coords)
        if check != []:
            print("Row Mismatch", filename2)
            if check[0] == -99:
                    #ids_ir = np.array(ir_coords.merge(raman_coords).index)
                    #ids_raman = np.array(raman_coords.merge(ir_coords).index)
                    
                    
                ir_coords_str = [str(x) + '_' + str(y) for x, y in zip(ir_coords['map_x'], ir_coords['map_y'])]
                raman_coords_str = [str(x) + '_' + str(y) for x, y in zip(raman_coords['map_x'], raman_coords['map_y'])]
                    
                ids_ir = np.where([x in raman_coords_str for x in ir_coords_str])[0]
                ids_raman = np.where([x in ir_coords_str for x in raman_coords_str])[0]
    
    
                ir_data = ir_data.loc[ids_ir, :].reset_index(drop=True)
                ir_coords = ir_coords.loc[ids_ir, :].reset_index(drop=True)
                ir_labels = list(np.array(ir_labels)[ids_ir])
                    
                raman_data = raman_data.loc[ids_raman, :].reset_index(drop=True)
                raman_coords = raman_coords.loc[ids_raman, :].reset_index(drop=True)
                raman_labels = list(np.array(raman_labels)[ids_raman])
            else:
                ir_data = ir_data.drop(check).reset_index(drop=True)
                ir_coords = ir_coords.drop(check).reset_index(drop=True)
                ir_labels = list(pd.DataFrame(ir_labels)[0].drop(check))
                    
                raman_data = raman_data.drop(check).reset_index(drop=True)
                raman_coords = raman_coords.drop(check).reset_index(drop=True)
                raman_labels = list(pd.DataFrame(raman_labels)[0].drop(check))
                
        #else:
            ###Set args for spectral type, range min, range max, and interpolate interval
            ###Run NA checks before merging
        ir_1, raman_1, raman_labels, ir_coords_2 = sp.remove_na_rows(ir_data, raman_data, raman_labels, ir_coords)
        if (ir_1.shape[0]!= ir_data.shape[0] or raman_1.shape[0]!= raman_data.shape[0]):
            print("NA Rows Removed but no other issues", filename2)
                
        args.spectral_type, args.range_min = "Raman", args.range_min_raman
        args.range_max, args.interpolate_interval = args.range_max_raman, args.interpolate_interval_raman
        ###The order within this function is: (1) Perform Baseline correction, (2) Perform Nitrogen Peak Calibration, 
        ##(3) Perform Silicon Peak Calibration, (4) Interpolate, and (5) export to the filename 
        #raman_2, XX_raman, rng, nitro_cal, saturated_wn = preprocess_data_raman(args, raman_1, raman_export_filename, args.si_peak[i])
        raman_2, XX_raman, rng, nitro_cal, saturated_wn = raman_1, raman_1.columns, 0, None, []
        ###Update args for spectral type, range min, range max, and interpolate interval
        args.spectral_type, args.range_min = "IR", args.range_min_ir
        args.range_max, args.interpolate_interval = args.range_max_ir, args.interpolate_interval_ir
        ###The order within this function is: (1) Interpolate, (2) subset to omit the middle region of data that isn't 
        ###collected by M4AM, and (4) export to the filename 
        ###Don't need to use any of the saved arguments from the function; those are just there for use in later steps,
        ###and so that you don't have to see all of the input printing to the screen. 
        #ir_2, XX_ir, rng, ir_bg_label = preprocess_data_ir(args, ir_1, ir_export_filename, args.wn_range_ir_1, args.wn_range_ir_2, args.si_peak[i])
        ir_2, XX_ir, rng, ir_bg_label = ir_1, ir_1.columns, 0, None 
        
        if merge_frames_exist == False:
            raman_merge_frame = np.zeros((0, len(XX_raman)))
            ir_merge_frame = np.zeros((0, len(XX_ir)))
            ir_bg_label_merge_frame = np.zeros((0))
            merge_frames_exist = True
            
        ###If we don't interpolate, and there are differences in the # of wave numbers, we'll run into an issue here.
        ###If there are the same number of wave # but the wave #'s are different, the wave #'s of the 1st will be used.
        raman_merge_frame = np.concatenate((raman_merge_frame,raman_2)) 
        ir_merge_frame = np.concatenate((ir_merge_frame,ir_2)) 
        #ir_bg_label_merge_frame = np.concatenate((ir_bg_label_merge_frame,ir_bg_label)) 
        coord_list.append(ir_coords_2)
        label_list.append(raman_labels) ###the labels will be the same for both Raman & IR, so doesn't matter.
            
        nitro_cal_list.append(nitro_cal)
        saturated_wn_list = saturated_wn_list + saturated_wn
        file_names_list = file_names_list + [filename]*ir_2.shape[0]
    
    ###Flatten list of lists
    label_list_flat = [x for xs in label_list for x in xs]
    label_list_flat = [int(element) for element in label_list_flat]
    #export_preprocessed_data(ir_export_filename, label_list_flat, ir_merge_frame, XX_ir, transpose)
    #export_preprocessed_data(raman_export_filename, label_list_flat, raman_merge_frame, XX_raman, transpose)
    coords_merge_frame = pd.concat(coord_list, axis=0).reset_index(drop=True)

    return raman_merge_frame, ir_merge_frame, label_list_flat, XX_ir, XX_raman, file_names_list, coords_merge_frame, nitro_cal_list, rng, ir_bg_label_merge_frame, saturated_wn_list

In [24]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')

args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak = [520]*len(files_list)
args.si_peak

[520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520,
 520]

In [ ]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = ptir_to_data_frame(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

In [18]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U50')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name.shape

(8186,)

In [20]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir.to_csv('enviromental_data/processed_data/passive_data_ir_raw.csv')

In [26]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

In [27]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U50')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name.shape

(8186,)

In [47]:
pd.DataFrame({'file_name':[x.split('/')[-1][:-5] for x in ids],
 'n':1}).groupby('file_name').sum()

,n
file_name,
DC_passive_AprMay2024_G7_20240529,219
DC_passive_August2024_J10_20240910,203
DC_passive_December2024_H5_20241217,202
DC_passive_July2024_I9_20240806,223
DC_passive_June2024_I7_20240717,217
DC_passive_MarApr2024_C9_20240612,204
DC_passive_May2024_D2_20240702,99
DC_passive_May2024_D2_20240703,104
DC_passive_November2024_C5_20241113,180


In [41]:
#trunc_saturated_wn_list = [x[XX_raman[x]>600] for x in saturated_wn_list]
#trunc_saturated_wn_list

In [42]:
#plt.hist([x.shape[0] for x in trunc_saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in trunc_saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [43]:
#plt.hist([x.shape[0] for x in saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [44]:
#plt.hist([x.shape[0] for x in saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [28]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())

with open('enviromental_data/processed_data/passive_data_raman_saturation.json', 'w') as f:
    json.dump(saturated_wn_dict, f)

In [48]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir['background'] = ir_bg_label
preposs_data_ir#.to_csv('enviromental_data/processed_data/passive_data_ir.csv')

,981.0,983.0,985.0,987.0,989.0,991.0,993.0,995.0,997.0,999.0,...,2959.0,2961.0,2963.0,2965.0,2967.0,2969.0,map_x,map_y,id,background
0,1.197118,1.098721,1.188965,1.155778,1.446818,1.745816,1.811037,1.843259,1.868581,1.955140,...,-1.019661,-0.737371,-1.247913,-1.289432,-1.049747,-1.429936,15435.200195,1589.199951,FHS_passive_August2024_B4_20240910_1,no_background
1,1.814813,1.851585,1.781849,1.807652,1.874676,1.787263,1.693712,1.723409,1.823371,1.715800,...,-1.800294,-2.237806,-1.659652,-0.313176,0.625383,0.643669,15436.75,1599.699951,FHS_passive_August2024_B4_20240910_2,no_background
2,1.850273,1.767589,1.600542,1.527601,1.734997,1.929096,1.909023,1.798808,1.460122,1.326416,...,-0.469963,-1.614735,-2.390781,-1.573574,-1.395273,-0.869554,15392.150391,1603.900024,FHS_passive_August2024_B4_20240910_3,no_background
3,0.637028,0.872861,1.397150,1.752766,1.792767,1.588734,1.623249,1.903908,2.087736,1.931579,...,-0.260542,-0.840206,-1.356421,-0.378981,-0.406569,-1.439772,15437.599609,1606.650024,FHS_passive_August2024_B4_20240910_4,no_background
4,2.326319,2.172114,1.328730,1.536732,1.910354,1.701816,1.543064,1.479553,1.686473,2.030196,...,-0.978635,-0.822985,-0.766489,-0.724351,-0.156693,0.292585,15439.450195,1611.050049,FHS_passive_August2024_B4_20240910_5,no_background
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8181,0.967205,1.239534,1.840630,1.992764,2.105520,2.556913,2.661188,2.526966,2.379944,2.158553,...,-0.164704,-0.680988,-1.066435,-1.384424,-0.788801,0.061634,7783.100098,-485.600006,IAO_passive_June2024_G7_20240801_39,no_background
8182,-0.197138,-0.126808,-0.104003,-0.160001,-0.162231,-0.142294,-0.146978,-0.119082,-0.097044,-0.111106,...,-0.572240,-0.667042,-0.819556,-0.756578,-0.774456,-0.810842,7778.299805,-487.200012,IAO_passive_June2024_G7_20240801_40,no_background
8183,1.909563,1.879518,1.898483,1.973327,1.839094,1.923762,2.137693,2.302368,2.398511,2.244184,...,-1.640979,-1.966160,-1.923280,-1.377506,-1.199033,-0.897168,7740.350098,-478.950012,IAO_passive_June2024_G7_20240801_41,no_background
8184,1.595697,2.059690,2.558675,2.728772,2.273458,1.956511,1.942726,1.688682,1.599504,1.937602,...,-0.531222,-1.024843,-1.751897,-1.714397,-0.813193,-0.816104,7828.049805,-470.100006,IAO_passive_June2024_G7_20240801_42,no_background


In [29]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman['wn_saturated'] = [x.shape[0] for x in saturated_wn_list]

preposs_data_raman_norm = preposs_data_raman.copy()
preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]
preposs_data_raman_norm.iloc[np.where(preposs_data_raman.iloc[:,:-4].std(1) == 0)[0], :-4] = 0

ids_no_signal = np.array(preposs_data_raman.query('id not in @ids_sat').iloc[np.where(np.sum(np.abs(np.diff(preposs_data_raman_norm.query('id not in @ids_sat').iloc[:,:-4], axis=1)), 1) < 8)[0],-2])
preposs_data_raman['no_signal'] = [(x in ids_no_signal)*1 for x in ids_file_name]
#preposs_data_raman = preposs_data_raman.loc[np.array(non_nas_idx), :].reset_index(drop=True)
preposs_data_raman['id_file'] = ['/'.join(x.split('/')[:-1]) for x in ids]
#preposs_data_raman.to_csv('enviromental_data/processed_data/passive_data_raman_specy.csv')
preposs_data_raman.to_csv('enviromental_data/processed_data/passive_data_raman_raw.csv')


/var/folders/9g/bj6gc6z17_1fyb7lx57xf04m0000gs/T/ipykernel_88543/1705111539.py:8: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  preposs_data_raman_norm.iloc[:,:-4] = (preposs_data_raman_norm.iloc[:,:-4] - preposs_data_raman_norm.iloc[:,:-4].mean(1)[:,None])/preposs_data_raman_norm.iloc[:,:-4].std(1)[:,None]


In [48]:
1

1